In [1]:
!pip install selenium pyarrow lxml requests beautifulsoup4 pandas
import time
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import os
import glob
import pickle
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.edge.options import Options

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36'
}

YEAR = 2020
SAVE_DIR = rf'C:\\Users\\lcsse\\Desktop\\STAGE_GENT\\scrapping\\data\\pcs{YEAR}'
os.makedirs(SAVE_DIR, exist_ok=True)

dates = []
list_of_result_dfs = []
races = []
data_by_month = {}


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


/Users/arthurdeletang/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
def is_page_not_found(response_text):
    return "Page not found" in response_text

def process_results_page(response_text, url):
    soup = BeautifulSoup(response_text, 'lxml')

    # Date
    try:
        date_tag = soup.select_one('div.value')
        date = date_tag.get_text(strip=True)
    except Exception:
        date = None
    dates.append(date)

    # Classification (2.UWT, 1.UWT, etc.)
    try:
        classif = None
        for li in soup.find_all('li'):
            if 'Classification:' in li.get_text():
                classif = li.get_text(strip=True).replace('Classification:', '').strip()
                break
    except Exception:
        classif = None

    # Startlist quality score
    try:
        startlist_quality = None
        for li in soup.find_all('li'):
            text = li.get_text()
            if 'Startlist quality score:' in text:
                match = re.search(r'Startlist quality score:\s*(\d+)', text)
                if match:
                    startlist_quality = int(match.group(1))
                break
    except Exception:
        startlist_quality = None

    # Average temperature
    try:
        avg_temperature = None
        for li in soup.find_all('li'):
            text = li.get_text()
            if 'Avg. temperature:' in text:
                match = re.search(r'Avg\. temperature:\s*([\d.]+)', text)
                if match:
                    avg_temperature = float(match.group(1))
                break
    except Exception:
        avg_temperature = None

    # Won how
    try:
        won_how = None
        for li in soup.find_all('li'):
            text = li.get_text()
            if 'Won how:' in text:
                won_how = text.split('Won how:')[-1].strip()
                break
    except Exception:
        won_how = None

    tables = soup.find_all('table')
    if tables:
        dfs = pd.read_html(str(tables))
        if dfs:
            df = dfs[0]
            df['date'] = date
            df['classification'] = classif
            df['startlist_quality'] = startlist_quality
            df['avg_temperature'] = avg_temperature
            df['won_how'] = won_how
            list_of_result_dfs.append(df)
            races.append(url)

def scrape_base_url(base_url):
    def has_results_table(response_text):
        soup = BeautifulSoup(response_text, 'lxml')
        return len(soup.find_all('table')) > 0

    result_url = base_url + "/result"
    response = requests.get(result_url, headers=headers)
    time.sleep(1)
    if has_results_table(response.text):
        process_results_page(response.text, result_url)

    stage = 1
    while True:
        stage_url = f"{base_url}/stage-{stage}"
        response = requests.get(stage_url, headers=headers)
        time.sleep(1)
        if has_results_table(response.text):
            process_results_page(response.text, stage_url)
            stage += 1
        else:
            break

    print(f"✅ {base_url}")

def scrape_urls(url_list):
    for base_url in url_list:
        scrape_base_url(base_url)

def clean_race_url(url):
    url = url.replace('/result', '')
    url = re.sub(r'/stage-\d+', '', url)
    return url

def save_month(month):
    for race, df, date in zip(data_by_month[month]['races'], data_by_month[month]['dfs'], data_by_month[month]['dates']):
        try:
            df['race_url'] = race
            df['date'] = date

            if '/stage-' in race:
                race_type = 'stage_' + race.split('/stage-')[-1]
            elif '/result' in race:
                race_type = 'result'
            else:
                race_type = 'one_day'

            race_name = race.replace('https://www.procyclingstats.com/race/', '')
            race_name = re.sub(r'/stage-\d+', '', race_name)
            race_name = race_name.replace('/result', '').replace('/', '_')

            clean_date = date.replace(' ', '_') if date else 'no_date'

            if 'classification' in df.columns and df['classification'].iloc[0]:
                classif = str(df['classification'].iloc[0]).strip()
            else:
                classif = 'no_classif'

            filename = f"{clean_date}__{race_type}__{race_name}__{classif}.csv"
            df.to_csv(os.path.join(SAVE_DIR, filename), index=False)
        except Exception as e:
            print(f"❌ {race}: {e}")
    print(f"✅ Mois {month}: {len(data_by_month[month]['races'])} fichiers sauvegardés")

In [3]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2020&month=1&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour janvier 2020")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[1] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(1)
driver.quit()

✅ 15 URLs trouvées pour janvier 2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/cambodia-bay-cycling-tour/2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-australia-itt/2020


...

✅ https://www.procyclingstats.com/race/nc-colombia-itt/2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/deia-trophy/2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/national-championships-qatar-itt/2020
Total brut: 45 | Uniques: 15 | Manquantes: 0
✅ Mois 1: 45 fichiers sauvegardés


In [4]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2020&month=2&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour fevrier 2020")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[2] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(2)
driver.quit()

✅ 36 URLs trouvées pour fevrier 2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/trofeo-pollenca-port-d-andratx/2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/great-ocean-road-race/2020


...

✅ https://www.procyclingstats.com/race/faun-ardeche-classic/2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/ster-van-zwolle/2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/grand-prix-velo-alanya/2020
Total brut: 94 | Uniques: 36 | Manquantes: 0
❌ https://www.procyclingstats.com/race/uae-tour/2020/stage-6: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/uae-tour/2020/stage-7: single positional indexer is out-of-bounds
✅ Mois 2: 94 fichiers sauvegardés


In [5]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2020&month=3&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour mars 2020")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[3] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(3)
driver.quit()

✅ 34 URLs trouvées pour mars 2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/la-drome-classic/2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/kuurne-brussel-kuurne/2020


...

✅ https://www.procyclingstats.com/race/classic-loire-atlantique/2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/cholet-pays-de-loire/2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/gp-adria-mobil/2020
Total brut: 50 | Uniques: 31 | Manquantes: 3
❌ https://www.procyclingstats.com/race/paris-nice/2020/stage-8: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/gp-industria-artigianato/2020/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/ronde-van-drenthe/2020/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/paris-troyes/2020/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/trofeu-da-arrabida/2020/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/nokere-koerse/2020/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/vuelta-ciclista-de-chile/2020/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/vuelta-ciclista-de-chile/2020/stage-2: single positional indexer 

In [6]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2020&month=4&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour avril 2020")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[4] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(4)
driver.quit()

✅ 25 URLs trouvées pour avril 2020
✅ https://www.procyclingstats.com/race/giro-di-sicilia/2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecate

✅ https://www.procyclingstats.com/race/joe-martin-stage-race-me/2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/route-adelie-de-vitre/2020


...

✅ https://www.procyclingstats.com/race/paris-mantes-en-yvelines/2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/east-midlands-international-cicle-classic/2020
✅ https://www.procyclingstats.com/race/tour-de-romandie/2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecate

✅ https://www.procyclingstats.com/race/tour-of-the-gila/2020
✅ https://www.procyclingstats.com/race/tour-de-yorkshire/2020
Total brut: 39 | Uniques: 18 | Manquantes: 7
❌ https://www.procyclingstats.com/race/joe-martin-stage-race-me/2020/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/joe-martin-stage-race-me/2020/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/joe-martin-stage-race-me/2020/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/joe-martin-stage-race-me/2020/stage-4: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/route-adelie-de-vitre/2020/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/volta-nxt-classic/2020/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/gran-premio-miguel-indurain/2020/result: single positional indexer is out-of-bo

In [7]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2020&month=5&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour mai 2020")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[5] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(5)
driver.quit()

✅ 47 URLs trouvées pour mai 2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/eschborn-frankfurt/2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/memorial-andrzeja-trochanowskiego/2020


...

✅ https://www.procyclingstats.com/race/gp-fany-gatroservis-pribram/2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/grand-prix-british-ophtalmology-center/2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/british-ophtalmology-center/2020
Total brut: 100 | Uniques: 45 | Manquantes: 2
❌ https://www.procyclingstats.com/race/eschborn-frankfurt/2020/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/memorial-andrzeja-trochanowskiego/2020/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/vuelta-asturias/2020/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/vuelta-asturias/2020/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/vuelta-asturias/2020/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/five-rings-of-moscow/2020/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/five-rings-of-moscow/2020/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/five-rings-of-moscow/2020/stag

In [8]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2020&month=6&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour juin 2020")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[6] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(6)
driver.quit()

✅ 21 URLs trouvées pour juin 2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/ronde-van-limburg/2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/trofeo-alcide-degasperi/2020


...

✅ https://www.procyclingstats.com/race/midden-brabant-poort-omloop/2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/gp-lugano/2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-slovenia-itt/2020
Total brut: 52 | Uniques: 20 | Manquantes: 1
❌ https://www.procyclingstats.com/race/ronde-van-limburg/2020/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/trofeo-alcide-degasperi/2020/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/ronde-de-l-oise/2020/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/ronde-de-l-oise/2020/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/ronde-de-l-oise/2020/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/ronde-de-l-oise/2020/stage-4: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/oberosterreichrundfahrt/2020/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/oberosterreichrundfahrt/2020/stage-2: single positional 

In [9]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2020&month=7&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour juillet 2020")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[7] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(7)
driver.quit()

✅ 28 URLs trouvées pour juillet 2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/trofeo-citta-di-brescia/2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/chrono-kristin-armstrong/2020


...

✅ https://www.procyclingstats.com/race/tour-of-bulgaria/2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecate

✅ https://www.procyclingstats.com/race/kreiz-breizh-elites/2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecate

✅ https://www.procyclingstats.com/race/tour-cycliste-international-de-la-guadeloupe/2020
Total brut: 59 | Uniques: 28 | Manquantes: 0
❌ https://www.procyclingstats.com/race/trofeo-citta-di-brescia/2020/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/chrono-kristin-armstrong/2020/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/grand-prix-erciyes-mimar-sinan-me/2020/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-de-delta/2020/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/grand-prix-kayseri-me/2020/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/gp-cerami/2020/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/challenge-du-prince-trophee-princier/2020/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/r

In [10]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2020&month=8&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour aout 2020")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[8] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(8)
driver.quit()

✅ 105 URLs trouvées pour aout 2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/strade-bianche/2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecate

✅ https://www.procyclingstats.com/race/la-route-d-occitanie/2020


...

✅ https://www.procyclingstats.com/race/brussels-cycling-classic/2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/gp-jef-scherens/2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/memorial-marco-pantani/2020
Total brut: 189 | Uniques: 105 | Manquantes: 0
❌ https://www.procyclingstats.com/race/antwerpse-havenpijl/2020/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/memorial-henryka-lasaka/2020/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/puchar-uzdrowisk-karpackich/2020/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-taiyuan/2020/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-taiyuan/2020/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-taiyuan/2020/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-taiyuan/2020/stage-4: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-taiyuan/2020/stage-5: single posit

In [11]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2020&month=9&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour septembre 2020")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[9] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(9)
driver.quit()

✅ 75 URLs trouvées pour septembre 2020
✅ https://www.procyclingstats.com/race/settimana-internazionale-coppi-e-bartali/2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/tour-de-serbie/2020
✅ https://www.procyclingstats.com/race/tour-of-denmark/2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecate

✅ https://www.procyclingstats.com/race/okolo-jiznich-cech/2020


...

✅ https://www.procyclingstats.com/race/renewi-tour/2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/la-fleche-wallonne/2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/coppa-agostoni/2020
Total brut: 131 | Uniques: 71 | Manquantes: 4
❌ https://www.procyclingstats.com/race/okolo-jiznich-cech/2020/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/okolo-jiznich-cech/2020/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/okolo-jiznich-cech/2020/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/okolo-jiznich-cech/2020/stage-4: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/okolo-jiznich-cech/2020/stage-5: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-kosovo/2020/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-kosovo/2020/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-kosovo/2020/stage-3: single positional indexer is o

In [12]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2020&month=10&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour octobre 2020")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[10] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(10)
driver.quit()

✅ 54 URLs trouvées pour octobre 2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/coppa-bernocchi/2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-ireland-itt/2020


...

✅ https://www.procyclingstats.com/race/popolarissima/2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecate

✅ https://www.procyclingstats.com/race/giro-di-sardegna/2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-switserland/2020
Total brut: 142 | Uniques: 54 | Manquantes: 0
❌ https://www.procyclingstats.com/race/coppa-bernocchi/2020/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-montenegro/2020/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-montenegro/2020/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-montenegro/2020/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/cyclassics-hamburg/2020/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/munsterland-giro/2020/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/famenne-ardenne-classic/2020/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/gp-bruno-beghelli/2020/result: single positional indexe

In [13]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2020&month=11&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour novembre 2020")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[11] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(11)
driver.quit()

✅ 17 URLs trouvées pour novembre 2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecate

✅ https://www.procyclingstats.com/race/vuelta-ciclista-a-venezuela/2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecate

✅ https://www.procyclingstats.com/race/tour-of-guangxi/2020


...

✅ https://www.procyclingstats.com/race/elite-road-central-american-championships-road-rac/2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-el-salvador/2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecate

✅ https://www.procyclingstats.com/race/vuelta-a-ecuador/2020
Total brut: 39 | Uniques: 17 | Manquantes: 0
❌ https://www.procyclingstats.com/race/vuelta-ciclista-a-venezuela/2020/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/vuelta-ciclista-a-venezuela/2020/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/vuelta-ciclista-a-venezuela/2020/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/vuelta-ciclista-a-venezuela/2020/stage-4: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/vuelta-ciclista-a-venezuela/2020/stage-5: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/vuelta-ciclista-a-venezuela/2020/stage-6: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/vuelta-ciclista-a-venezuela/2020/stage-7: single positional indexer is out-of-bounds
❌ https://www.procyclingst

In [14]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2020&month=12&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour decembre 2020")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[12] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(12)
driver.quit()

✅ 2 URLs trouvées pour decembre 2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-paraguay-itt/2020


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_12543/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-paraguay/2020
Total brut: 2 | Uniques: 2 | Manquantes: 0
✅ Mois 12: 2 fichiers sauvegardés
